# Business Intelligence & Data Engineering
---
**MSc in Business Analytics** | *Athens University of Economics and Business* **Assignment:** Apache Spark  
**Student:** Anastasios Rigos (f2822511)

Apache Spark run on Java 17 or Java 21, we will use Java 21.

In [2]:
import os

JAVA_HOME = "/opt/homebrew/Cellar/openjdk@21/21.0.10/libexec/openjdk.jdk/Contents/Home"
os.environ["JAVA_HOME"] = JAVA_HOME

In [3]:
! echo $JAVA_HOME

/opt/homebrew/Cellar/openjdk@21/21.0.10/libexec/openjdk.jdk/Contents/Home


In [4]:
! java -version

openjdk version "21.0.10" 2026-01-20
OpenJDK Runtime Environment Homebrew (build 21.0.10)
OpenJDK 64-Bit Server VM Homebrew (build 21.0.10, mixed mode, sharing)


# Part 1 - SparkSQL

We set up the scene by importing from the class SparkSession from the library pyspark  

In [5]:
from pyspark.sql import SparkSession

We create a Spark Session for the Q-Commerce project with Application Name "Q-Commerce-project".

In [6]:
spark = SparkSession.builder.appName("Q-Commerce-project").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/11 16:58:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/11 16:58:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


We read the retail-dataset.csv file in the folder ./data and we save it in a dataframe with the name retail_df.

In [7]:
# header=True: the first row of the CSV file contains the column names
# inferSchema=True: automatically infer the data types of the columns
retail_df = spark.read.csv("data/retail-dataset.csv", header=True, inferSchema=True)
retail_df.cache()

DataFrame[Transaction_ID: int, Date: timestamp, Customer_Name: string, Product: string, Total_Items: int, Total_Cost: double, Payment_Method: string, City: string, Store_Type: string, Discount_Applied: boolean, Customer_Category: string, Season: string, Promotion: string]

## Question 1: Display descriptive statistics for each column of the dataset.  

We display the descriptive statistics for each column of the dataset using the describe() method in pandas, which provides us the count, the meand, the standard deviation, the minimum and the maximum value of each column.

In [8]:
retail_df.describe().toPandas()

26/04/11 16:58:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,summary,Transaction_ID,Customer_Name,Product,Total_Items,Total_Cost,Payment_Method,City,Store_Type,Customer_Category,Season,Promotion
0,count,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000
1,mean,1.0004999995E9,NaN,NaN,5.495941,52.455220400000094,NaN,NaN,NaN,NaN,NaN,NaN
2,stddev,288675.2789323441,NaN,NaN,2.871654187209314,27.41698914533317,NaN,NaN,NaN,NaN,NaN,NaN
3,min,1000000000,Aaron Acevedo,"['Air Freshener', 'Air Freshener', 'Air Freshe...",1,5.0,Cash,Atlanta,Convenience Store,Homemaker,Fall,BOGO (Buy One Get One)
4,max,1000999999,Zoe York,['Yogurt'],10,100.0,Mobile Payment,Seattle,Warehouse Club,Young Adult,Winter,None


Alternatively, we can use the summary() method, which provide also the 25%, 50%, 75% quartiles for the numeric variables

In [9]:
retail_df.summary().toPandas()

,summary,Transaction_ID,Customer_Name,Product,Total_Items,Total_Cost,Payment_Method,City,Store_Type,Customer_Category,Season,Promotion
0,count,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000
1,mean,1.0004999995E9,NaN,NaN,5.495941,52.455220400000094,NaN,NaN,NaN,NaN,NaN,NaN
2,stddev,288675.2789323441,NaN,NaN,2.871654187209314,27.41698914533317,NaN,NaN,NaN,NaN,NaN,NaN
3,min,1000000000,Aaron Acevedo,"['Air Freshener', 'Air Freshener', 'Air Freshe...",1,5.0,Cash,Atlanta,Convenience Store,Homemaker,Fall,BOGO (Buy One Get One)
4,25%,1000249917,NaN,NaN,3,28.71,NaN,NaN,NaN,NaN,NaN,NaN
5,50%,1000500057,NaN,NaN,5,52.42,NaN,NaN,NaN,NaN,NaN,NaN
6,75%,1000749990,NaN,NaN,8,76.18,NaN,NaN,NaN,NaN,NaN,NaN
7,max,1000999999,Zoe York,['Yogurt'],10,100.0,Mobile Payment,Seattle,Warehouse Club,Young Adult,Winter,None


As we can see the dataset contains 1,000,000 rows of transcations. The numeric columns consist of Transaction_ID, Total_Items and Total_Cost and the categorical ones are Customer_Name, Product

## Question 2. Identify the customers that pay in cash and display the two corresponding columns.  

Let's see which are the types of the Payment Method.

In [10]:
# Select distinct values from the Payment_Method column and convert the result to a Pandas DataFrame
retail_df.select("Payment_Method").distinct().toPandas()

,Payment_Method
0,Mobile Payment
1,Credit Card
2,Cash
3,Debit Card


We see that that the Payment Method for paying in cash is "Cash".  
So we want to find the transactions that the customer paid with cash and display only the two corresponding columns.

In [11]:
# Filter the DataFrame to include only rows where the Payment_Method is 'Cash',
# select only the Customer_Name and Payment_Method columns 
# and convert the resulting DataFrame to a Pandas DataFrame
retail_df.filter("Payment_Method == 'Cash'") \
    .select("Customer_Name", "Payment_Method") \
    .toPandas()

,Customer_Name,Payment_Method
0,Michelle Carlson,Cash
1,Joshua Frazier,Cash
2,Victoria Garrett,Cash
3,Angela Jones,Cash
4,Don Harris,Cash
...,...,...
250225,Joshua Green,Cash
250226,Leslie Sanford,Cash
250227,Gary Mosley,Cash
250228,James Villegas,Cash


We have 250230 transactions, where customers paid in cash with their corresponding names.

## Question 3: Identify and display the purchases carried out by students or teenagers.  

Let's see which values in Customer_Category column corresponds to students and teenagers

In [12]:
# Find the distinct values in the Customer_Category column.
retail_df.select("Customer_Category").distinct().toPandas()

,Customer_Category
0,Teenager
1,Student
2,Retiree
3,Homemaker
4,Senior Citizen
5,Young Adult
6,Professional
7,Middle-Aged


So, we want to filder the dataset and keep only the purschases having Customer_Category value of "Teemager" or "Student"

In [13]:
# Filter the DataFrame to include only rows where the Customer_Category is 'Teenager' or 'Student'
retail_df.filter("Customer_Category == 'Teenager' or Customer_Category == 'Student'") \
    .toPandas()

,Transaction_ID,Date,Customer_Name,Product,Total_Items,Total_Cost,Payment_Method,City,Store_Type,Discount_Applied,Customer_Category,Season,Promotion
0,1000000006,2023-01-08 10:40:03,Victoria Garrett,"['Honey', 'BBQ Sauce', 'Soda', 'Olive Oil', 'G...",4,5.28,Cash,Boston,Specialty Store,False,Student,Summer,Discount on Selected Items
1,1000000012,2023-05-27 15:52:59,Sheila Mcgee,"['Salmon', 'Shaving Cream']",9,39.75,Debit Card,New York,Pharmacy,False,Student,Summer,Discount on Selected Items
2,1000000020,2021-10-31 07:46:22,Angela Jones,"['Pancake Mix', 'Vacuum Cleaner']",7,83.86,Cash,Houston,Pharmacy,False,Teenager,Winter,Discount on Selected Items
3,1000000024,2023-02-27 16:46:56,Hannah Rowland,"['Sponges', 'Vacuum Cleaner', 'Iron']",4,24.03,Debit Card,Dallas,Pharmacy,False,Student,Winter,None
4,1000000028,2022-08-13 01:26:19,Clayton Roth,"['Tomatoes', 'Salmon']",7,21.08,Credit Card,San Francisco,Specialty Store,True,Teenager,Fall,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
250156,1000999972,2020-05-24 19:16:30,William Brown,['Shaving Cream'],8,32.64,Credit Card,Houston,Specialty Store,False,Teenager,Fall,Discount on Selected Items
250157,1000999977,2023-10-05 09:53:09,Robin Waters,"['Tuna', 'Orange', 'Ketchup']",8,87.59,Mobile Payment,Los Angeles,Supermarket,True,Student,Summer,BOGO (Buy One Get One)
250158,1000999983,2023-10-02 22:34:12,Mr. James Buchanan,"['Paper Towels', 'Tomatoes']",2,74.41,Credit Card,Atlanta,Warehouse Club,False,Student,Spring,BOGO (Buy One Get One)
250159,1000999988,2023-06-08 01:04:43,Blake Richard,"['Chips', 'Razors', 'Tea', 'Tomatoes']",9,94.09,Mobile Payment,Atlanta,Convenience Store,False,Student,Summer,BOGO (Buy One Get One)


## Question 4: Identify and display the customer categories included in the dataset and the number of purchases per customer category.  

As we did for the cell above, we find the customer categories by finding the unique/distict values of the category Customer_Category column.

In [14]:
# Select the Customer_Category column and find the distinct values of the column.
retail_df.select("Customer_Category").distinct().toPandas()

,Customer_Category
0,Teenager
1,Student
2,Retiree
3,Homemaker
4,Senior Citizen
5,Young Adult
6,Professional
7,Middle-Aged


We have eight different customer categories, as we can see from the Dataframe. 

- Teenager
- Student 
- Retiree
- Homemaker 
- Senior Citizen
- Young Adult 
- Professional
- Middle-Aged

Let's now find the the number of purchases per customer category. 

In [15]:
# Group the DataFrame by the Customer_Category column,
# count the number of occurrences of each category,
# sort the results in descending order based on the count 
# and convert the result to a Pandas DataFrame
retail_df.groupBy("Customer_Category") \
    .count() \
    .sort("count", ascending=False) \
    .toPandas()

,Customer_Category,count
0,Senior Citizen,125485
1,Homemaker,125418
2,Teenager,125319
3,Retiree,125072
4,Student,124842
5,Professional,124651
6,Middle-Aged,124636
7,Young Adult,124577


We can see from the Datadrame the number of purchses for every customer category, with Senior Citizens having the most purchses in the datasets with 125485 transactions, while the sutomer category with the least purchases (124,577) are the Young Adults.

## Question 5: Identify the purchases that included a promotion and display the 10 most recent ones.  

We have find which are the different types of the promotions.

In [16]:
# Find the different promotions in the Promotion column
retail_df.select("Promotion").distinct().toPandas()

,Promotion
0,BOGO (Buy One Get One)
1,None
2,Discount on Selected Items


The Promotion column containd 3 unique values: 
- BOGO (Buy One Get One)
- None
- Discount on Selected Items
So, we have to filter the dataset to include only the rows with Promotion BOGO (Buy One Get One) or Discount on Selected Items or we can exlude the rows when Promotion equals None.

We will filter the dataset to include only the rows with Promotion not None.

In [17]:
# Filter the DataFrame to include only rows where the Promotion is not 'None' 
# (i.e., there is a promotion applied to the purchase)
# and display the last 10 rows of the resulting DataFrame as a Pandas DataFrame
retail_df.filter("Promotion != 'None'").toPandas().iloc[-10:]

,Transaction_ID,Date,Customer_Name,Product,Total_Items,Total_Cost,Payment_Method,City,Store_Type,Discount_Applied,Customer_Category,Season,Promotion
666047,1000999982,2023-02-12 02:16:32,Leslie Sanford,['Trash Cans'],4,66.85,Cash,Miami,Pharmacy,True,Middle-Aged,Summer,BOGO (Buy One Get One)
666048,1000999983,2023-10-02 22:34:12,Mr. James Buchanan,"['Paper Towels', 'Tomatoes']",2,74.41,Credit Card,Atlanta,Warehouse Club,False,Student,Spring,BOGO (Buy One Get One)
666049,1000999987,2020-07-23 10:27:00,Shaun Wilkerson,"['Pancake Mix', 'Pasta']",9,85.92,Mobile Payment,New York,Warehouse Club,True,Young Adult,Summer,Discount on Selected Items
666050,1000999988,2023-06-08 01:04:43,Blake Richard,"['Chips', 'Razors', 'Tea', 'Tomatoes']",9,94.09,Mobile Payment,Atlanta,Convenience Store,False,Student,Summer,BOGO (Buy One Get One)
666051,1000999990,2020-03-18 19:54:23,Krista Miller,"['Jam', 'Light Bulbs', 'Mayonnaise']",10,96.19,Credit Card,Los Angeles,Supermarket,False,Retiree,Fall,BOGO (Buy One Get One)
666052,1000999993,2022-01-02 19:40:32,Donna Hamilton,"['Laundry Detergent', 'Feminine Hygiene Produc...",6,65.29,Debit Card,Dallas,Specialty Store,True,Student,Summer,BOGO (Buy One Get One)
666053,1000999994,2021-03-13 14:12:17,James Villegas,"['Toothbrush', 'Hair Gel', 'Milk']",8,99.12,Cash,Los Angeles,Supermarket,False,Retiree,Summer,BOGO (Buy One Get One)
666054,1000999996,2022-05-19 05:13:58,Emily Graham,['Cereal'],8,80.25,Cash,Houston,Warehouse Club,True,Senior Citizen,Spring,Discount on Selected Items
666055,1000999998,2023-10-17 05:50:40,Michael Rodriguez,"['Diapers', 'Coffee', 'Coffee', 'Mop']",3,23.48,Debit Card,San Francisco,Supermarket,True,Retiree,Winter,BOGO (Buy One Get One)
666056,1000999999,2020-06-15 11:58:49,Jennifer Davis,"['Trash Cans', 'Mop', 'Jam']",8,44.12,Credit Card,Atlanta,Pharmacy,False,Professional,Fall,Discount on Selected Items


## Question 6: Find the distinct cities where purchases have occured and order the cities by the number of purchases that have happened in each city from highest to lowest.  

We have to group the dataset by the different cities, count the number of purchases for every city and then sort the cities in descending order of the number of purchases.

In [18]:
# Group the DataFrame by the City column,
# count the number of occurrences of each city,
# sort the results in descending order based on the count
# and convert the result to a Pandas DataFrame
retail_df.groupBy("City") \
    .count() \
    .sort("count", ascending=False) \
    .toPandas()

,City,count
0,Boston,100566
1,Dallas,100559
2,Seattle,100167
3,Chicago,100059
4,Houston,100050
5,New York,100007
6,Los Angeles,99879
7,Miami,99839
8,San Francisco,99808
9,Atlanta,99066


As we can see from the output, purchases occured in 10 different/distict cities with Boston leading with 100,566 purchases.

In [19]:
retail_df.columns

['Transaction_ID',
 'Date',
 'Customer_Name',
 'Product',
 'Total_Items',
 'Total_Cost',
 'Payment_Method',
 'City',
 'Store_Type',
 'Discount_Applied',
 'Customer_Category',
 'Season',
 'Promotion']

Question 7

In [42]:
import pyspark.sql.functions as fs

retail_df.groupBy(fs.to_date(retail_df.Date).alias("Date")) \
    .count().withColumnRenamed("count", "Number_of_Purchases") \
    .sort("Number_of_Purchases", ascending=False) \
    .toPandas().loc[:10]

,Date,Number_of_Purchases
0,2022-06-07,706
1,2021-03-21,696
2,2021-01-04,694
3,2024-02-02,694
4,2021-03-03,694
5,2022-05-31,693
6,2021-03-25,691
7,2023-03-30,690
8,2021-10-03,685
9,2022-09-21,685


question 8

In [43]:
# Filter 
retail_df.filter("City == 'Los Angeles'") \
    .groupBy("Store_Type") \
    .mean("Total_Cost") \
    .toPandas()

,Store_Type,avg(Total_Cost)
0,Warehouse Club,52.272274
1,Supermarket,52.566340
2,Department Store,52.356963
3,Convenience Store,52.262503
4,Pharmacy,52.715907
5,Specialty Store,52.150182


question 9

In [44]:
# Filter the DataFrame to include only rows where the Customer_Category is 'Professional',
# group the resulting DataFrame by the City and Customer_Name columns
# calculate the sum of the Total_Cost column for each group,
# (i.e., for each combination of City and Customer_Name), 
# sort the results first by City in ascending order and then by the sum of Total_Cost in descending order,
# drop duplicate rows based on the City column (i.e., keep only one row for each city)
# and convert the final result to a Pandas DataFrame
retail_df.filter("Customer_Category == 'Professional'") \
    .groupBy("City", "Customer_Name") \
    .sum("Total_Cost") \
    .orderBy("City", "sum(Total_Cost)", ascending=[True, False]) \
    .drop_duplicates(["City"]) \
    .toPandas()

,City,Customer_Name,sum(Total_Cost)
0,Atlanta,Nicholas Smith,336.50
1,Boston,Kevin Smith,551.64
2,Chicago,Michael Smith,520.93
3,Dallas,Sarah Johnson,407.96
4,Houston,David Johnson,380.64
5,Los Angeles,Michael Martin,505.82
6,Miami,Robert Smith,451.32
7,New York,Michael Smith,418.82
8,San Francisco,Michael Jones,367.57
9,Seattle,Michael Smith,445.31


# Part 2 - MLlib

In [45]:
retail_df.printSchema()

root
 |-- Transaction_ID: integer (nullable = true)
 |-- Date: timestamp (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Total_Items: integer (nullable = true)
 |-- Total_Cost: double (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Store_Type: string (nullable = true)
 |-- Discount_Applied: boolean (nullable = true)
 |-- Customer_Category: string (nullable = true)
 |-- Season: string (nullable = true)
 |-- Promotion: string (nullable = true)



In [46]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

categoricalCols = ["Customer_Category", "Store_Type", "City"]

stringIndexer = StringIndexer(inputCols=categoricalCols, outputCols=[col + "_Index" for col in categoricalCols])
encoder = OneHotEncoder(inputCols=stringIndexer.getOutputCols(), outputCols=[col + "_OHE" for col in categoricalCols])

In [47]:
from pyspark.ml.feature import VectorAssembler

# This includes both the numeric columns and the one-hot encoded binary vector columns in our dataset.
numericCols = ["Total_Cost"]
assemblerInputs = [c + "_OHE" for c in categoricalCols] + numericCols
vecAssembler = VectorAssembler(inputCols=assemblerInputs, outputCol="features")

Build the pipeline

In [48]:
from pyspark.ml import Pipeline

transformation_pipeline = Pipeline()\
  .setStages([stringIndexer, encoder, vecAssembler])
fitted_pipeline = transformation_pipeline.fit(retail_df)

In [ ]:
transformed_dataframe = fitted_pipeline.transform(retail_df)
transformed_dataframe.cache()
transformed_dataframe.toPandas()

ConnectionRefusedError: [Errno 61] Connection refused

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

for i in range(2, 6):
    kmeans = KMeans().setK(i).setSeed(1)
    model = kmeans.fit(transformed_dataframe)
    predictions = model.transform(transformed_dataframe)
    evaluator = ClusteringEvaluator()
    silhouette = evaluator.evaluate(predictions)
    print("["+str(i)+":] Silhouette with squared euclidean distance = " + str(silhouette))